# Week 4 — ETL, OLAP, Medallion (Advanced Notebook)

## 🔥 What You Will Learn
This notebook is a **complete end-to-end data pipeline**:

- Raw messy data (Bronze)
- Cleaned data (Silver)
- Business metrics (Gold)
- OLAP queries
- Visualization + business insight

👉 Every section includes:
- Explanation
- SQL
- Python
- Business meaning


## Step 0 — Setup

We use:
- DuckDB (analytical database)
- Pandas (inspection)
- Matplotlib (visualization)

👉 Why this matters:
Modern analysts combine SQL + Python


In [ ]:

%load_ext sql
%sql duckdb:///week04_pipeline_v2.duckdb

import pandas as pd
import matplotlib.pyplot as plt
import duckdb

con = duckdb.connect("week04_pipeline_v2.duckdb")


## Step 1 — Create Raw Data (Bronze Input)

### What we are doing
We simulate messy real-world data:
- inconsistent casing
- bad values
- mixed types

### Business Insight
👉 Real data is NEVER clean
👉 Pipelines exist because of this


In [ ]:

raw = pd.DataFrame({
    "order_id":[1,2,3,4,5,6],
    "region":["West ","west","EAST","East ","west","WEST"],
    "amount":["1000","2000","1500","ERROR","1800","1700"]
})

raw.to_csv("raw_sales.csv", index=False)
raw


## Step 2 — Bronze Layer

### What we are doing
- Load raw data as-is
- No cleaning

### Business Insight
👉 Bronze = source of truth
👉 Used for debugging + traceability


In [ ]:

%sql DROP TABLE IF EXISTS bronze_sales;
%sql CREATE TABLE bronze_sales AS SELECT * FROM read_csv_auto('raw_sales.csv');

%sql SELECT * FROM bronze_sales;


## Step 3 — Data Profiling

### What we are doing
- Check nulls
- Understand data issues

### Business Insight
👉 Always profile BEFORE transforming
👉 Prevents bad assumptions


In [ ]:

res = %sql SELECT COUNT(*) total_rows, COUNT(amount) non_null_amount FROM bronze_sales
res.DataFrame()


In [ ]:

df = pd.read_csv("raw_sales.csv")
df


## Step 4 — Silver Layer (Cleaning)

### What we are doing
- Clean region values
- Convert amount to numeric
- Remove bad rows

### Business Insight
👉 Silver = trusted data
👉 This is where most logic lives


In [ ]:

%sql DROP TABLE IF EXISTS silver_sales;

%sql CREATE TABLE silver_sales AS
SELECT
    order_id,
    LOWER(TRIM(region)) AS region,
    TRY_CAST(amount AS DOUBLE) AS amount
FROM bronze_sales
WHERE TRY_CAST(amount AS DOUBLE) IS NOT NULL;

%sql SELECT * FROM silver_sales;


## Step 5 — Verify Silver (Python)

### What we are doing
- Inspect cleaned dataset

### Business Insight
👉 Always validate transformation
👉 Never assume correctness


In [ ]:

silver_df = con.execute("SELECT * FROM silver_sales").fetchdf()
silver_df


## Step 6 — Gold Layer (Business Metrics)

### What we are doing
- Aggregate revenue by region

### Business Insight
👉 Gold = decision layer
👉 Executives care about THIS


In [ ]:

%sql DROP TABLE IF EXISTS gold_sales;

%sql CREATE TABLE gold_sales AS
SELECT
    region,
    SUM(amount) AS total_revenue
FROM silver_sales
GROUP BY region;

%sql SELECT * FROM gold_sales;


## Step 7 — Visualization

### What we are doing
- Plot revenue by region

### Business Insight
👉 Visuals reveal patterns faster than tables


In [ ]:

gold_df = con.execute("SELECT * FROM gold_sales").fetchdf()

plt.figure()
plt.bar(gold_df["region"], gold_df["total_revenue"])
plt.title("Revenue by Region")
plt.xlabel("Region")
plt.ylabel("Revenue")
plt.grid(axis="y", alpha=0.3)
plt.show()


## Step 8 — OLAP Query (Slice)

### What we are doing
- Analyze one segment

### Business Insight
👉 Slice = focused decision-making


In [ ]:

%sql SELECT region, SUM(amount) FROM silver_sales GROUP BY region;


## Step 9 — Validation

### What we are doing
- Check for anomalies

### Business Insight
👉 Trust but verify


In [ ]:

%sql SELECT * FROM silver_sales WHERE amount < 0;


## 🎯 Final Summary

- Bronze = raw data
- Silver = cleaned data
- Gold = business metrics
- OLAP = insights

👉 This is how real data systems work
